# Spot model calibration on OMIE day-ahead Spain

Phase 2 / Pieza 1 of the mibel-derivatives module.

Mean-reverting jump-diffusion on the hourly UTC log-price, with a
deterministic seasonal component (Fourier annual + day-of-week +
hour-of-day) and Kou asymmetric double-exponential jumps. Spec and
decisions recorded in `src/mibel_derivatives/models/spot.py` and
`reports/diagnostics/spot_model_calibration.md`.


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sstats
from statsmodels.tsa.stattools import adfuller

from mibel_derivatives.models import spot

pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 110


## 1. Load curated data

Manus-sourced OMIE day-ahead Spain via ESIOS indicator 600. UTC
hourly granularity 2019-01-01 → 2024-12-31. Filter to on-the-hour
observations (the source contains three trailing 15-min ticks at
boundaries that we drop).

In [ ]:
df = pd.read_parquet('data/curated/omie_spot_es_2019_2024.parquet')
df = df.set_index('datetime_utc').sort_index()
hourly = df[df.index.minute == 0]['price_eur_mwh'].rename('price_eur_mwh')
hourly.index.name = 'dt_utc'
print('Rows:', len(hourly))
print('Coverage:', hourly.index.min(), '..', hourly.index.max())
print('Stats:', hourly.describe())


## 2. Exploratory analysis

- Full hourly series 2019-2024 with the 2022 gas-crisis spike.
- Histogram of nominal prices (heavy right tail).
- Autocorrelation of one-hour log-returns.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hourly.index, hourly.values, lw=0.3)
ax.set_title('OMIE day-ahead Spain (ESIOS 600), 2019-2024 hourly')
ax.set_ylabel('EUR/MWh'); ax.set_xlabel('UTC date'); ax.grid(alpha=0.3)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(hourly.values, bins=100, color='steelblue')
axes[0].set_title('Hourly price distribution (EUR/MWh)')
axes[0].grid(alpha=0.3)
log_returns = np.log(hourly + 10).diff().dropna()
lags = list(range(1, 49))
acf = [log_returns.autocorr(lag=k) for k in lags]
axes[1].bar(lags, acf, color='darkorange')
axes[1].set_title('Autocorrelation of log(P+10) returns up to 48 h')
axes[1].set_xlabel('lag (hours)'); axes[1].grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()


## 3. Calibration

`spot.fit` runs the four-step pipeline: OLS for the seasonal
component → residuals → iterative threshold jump detection at k=4 →
MLE for OU (κ, σ_h) and Kou (λ, p_up, η_up, η_down).

In [ ]:
fit = spot.fit(hourly)
p = fit.params
print(f'intercept       : {p.seasonality.intercept:.4f}')
print(f'fourier coefs   : {np.round(p.seasonality.fourier_coefs, 4)}')
print(f'DoW coefs       : {np.round(p.seasonality.dow_coefs, 4)}')
print(f'HoD coefs range : {p.seasonality.hod_coefs.min():+.4f}..{p.seasonality.hod_coefs.max():+.4f}')
print(f'kappa           : {p.kappa:.5f} /h  (half-life {np.log(2)/p.kappa:.1f} h)')
print(f'sigma_h         : {p.sigma_by_hour.min():.4f}..{p.sigma_by_hour.max():.4f}  (mean {p.sigma_by_hour.mean():.4f})')
print(f'jump intensity  : {p.jump_intensity:.5f}/h  ({p.jump_intensity*8760:.0f}/year)')
print(f'jump p_up       : {p.jump_p_up:.3f}')
print(f'jump eta_up     : {p.jump_eta_up:.3f}  (mean +J {1/p.jump_eta_up:.3f} log)')
print(f'jump eta_down   : {p.jump_eta_down:.3f}  (mean -J {1/p.jump_eta_down:.3f} log)')
print(f'n_obs={fit.n_obs}  n_jumps={fit.n_jumps} ({100*fit.n_jumps/fit.n_obs:.2f} %)')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
doys = np.arange(1, 366); ang = 2*np.pi*doys/365.25
fy = np.zeros_like(doys, dtype=float)
for kh in range(1, 5):
    fy = fy + p.seasonality.fourier_coefs[2*(kh-1)] * np.cos(kh*ang)
    fy = fy + p.seasonality.fourier_coefs[2*kh - 1] * np.sin(kh*ang)
axes[0].plot(doys, fy)
axes[0].set_title('Fourier annual component'); axes[0].set_xlabel('day of year')
axes[0].grid(alpha=0.3)
dow_full = np.concatenate([[0.0], p.seasonality.dow_coefs])
axes[1].bar(range(7), dow_full,
            tick_label=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])
axes[1].set_title('Day-of-week dummies'); axes[1].grid(alpha=0.3, axis='y')
hod_full = np.concatenate([[0.0], p.seasonality.hod_coefs])
axes[2].bar(range(24), hod_full)
axes[2].set_title('Hour-of-day dummies (UTC)'); axes[2].set_xlabel('hour UTC')
axes[2].grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(24), p.sigma_by_hour)
ax.set_title(f'OU sigma by hour-of-day (UTC); kappa={p.kappa:.4f}/h')
ax.set_xlabel('hour UTC'); ax.set_ylabel('sigma'); ax.grid(alpha=0.3, axis='y')
plt.show()


## 4. Residual diagnostics

- ADF on the residual level (stationarity).
- ADF on the residual returns (sanity check that returns are stationary).
- Jarque-Bera on non-jump returns (normality of the OU innovation,
  expected to be REJECTED — kurtosis is heavy even after jump removal,
  the documented limitation of a single-factor MRJD on 2019-2024).

In [ ]:
nj = fit.residual_returns[~fit.jumps_mask]
adf_resid = adfuller(fit.residuals.values, regression='c', autolag='AIC')
adf_ret = adfuller(fit.residual_returns.values, regression='c', autolag='AIC')
jb = sstats.jarque_bera(nj.values)
print(f'ADF residuals   : stat={adf_resid[0]:.3f}  p={adf_resid[1]:.3g}')
print(f'ADF returns     : stat={adf_ret[0]:.3f}  p={adf_ret[1]:.3g}')
print(f'JB non-jump ret : stat={jb.statistic:.1f}  p={jb.pvalue:.3g}')
print(f'  return std all     : {fit.residual_returns.std():.4f}')
print(f'  return std nonjump : {nj.std():.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(fit.residuals.index, fit.residuals.values, lw=0.3, label='residuals')
jr_idx = fit.residual_returns.index[fit.jumps_mask.values]
ax.scatter(jr_idx, fit.residuals.reindex(jr_idx).values,
           s=4, color='red', label=f'jumps n={fit.n_jumps}', zorder=3)
ax.set_title('Deseasonalised residuals Z_t with detected jumps')
ax.set_ylabel('log(P+10) residual'); ax.grid(alpha=0.3); ax.legend()
plt.show()


## 5. Forward simulation and validation

Simulate 100 paths over 1 year (2025) from the end-of-2024 residual
state. Compare path-level statistics to the historical 2019-2024:
mean, std, hourly autocorrelation, frequency of large moves.

In [ ]:
sim_start = hourly.index[-1] + pd.Timedelta('1h')
last_resid = float(fit.residuals.iloc[-1])
paths = spot.simulate(p, sim_start, n_hours=24*365, n_paths=100,
                      initial_residual=last_resid, seed=2026)
print(f'Simulated mean : {paths.mean():.2f} EUR/MWh')
print(f'Simulated std  : {paths.std():.2f}')
print(f'Historical mean: {hourly.mean():.2f}')
print(f'Historical std : {hourly.std():.2f}')

sim_log_returns = np.diff(np.log(paths + p.price_shift), axis=1).ravel()
hist_log_returns = np.log(hourly + p.price_shift).diff().dropna().values
print(f'sim   return std={sim_log_returns.std():.4f}')
print(f'hist  return std={hist_log_returns.std():.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sim_idx = pd.date_range(sim_start, periods=paths.shape[1], freq='h')
ax.fill_between(sim_idx, np.percentile(paths, 10, axis=0),
                np.percentile(paths, 90, axis=0), alpha=0.2,
                label='P10-P90 sim')
ax.plot(sim_idx, paths.mean(axis=0), label='mean sim', lw=0.6)
ax.set_title('Simulated 2025 paths (n=100) from end-2024 state')
ax.set_ylabel('EUR/MWh'); ax.grid(alpha=0.3); ax.legend()
plt.show()


## 6. Notes carried forward

Concrete limitations documented in
`reports/diagnostics/spot_model_calibration.md`:

1. Single 2019-2024 calibration mixes the pre-crisis and post-crisis
   regime; ADF rejects unit root for both residuals and returns but
   the equilibrium level is non-stationary in level. The long-term
   factor of the Schwartz-Smith piece (Pieza 2) is intended to absorb
   this.
2. The simulated 1-year dispersion is much larger than the
   historical because the slow mean reversion (half-life ≈ 100 h)
   compounds OU + jumps over a long horizon. Short-horizon
   simulation (days to a few weeks) tracks historical statistics
   much more closely.
3. At k=4 the iterative threshold flags ~3.6 % of returns as jumps,
   high vs the 1-2 % typical in the European-power MRJD literature;
   this is partly the 2022 spike regime and partly the
   heavy-tailed innovations remaining after deseasonalisation.
